# Step 1 · Hello, local LLM
One prompt to a model running **on this machine**, sent through the **Bifrost** gateway and traced in **Langfuse**.
Nothing leaves the laptop.

> **Start Jupyter through the project script** so it gets the same settings as the app:
>
> ```bash
> cd copilot
> make notebook
> ```

Needs: the services running (`make up`) with the models pulled (`make setup`) for Bifrost + Langfuse.

In [ ]:
import sys, os
sys.path.insert(0, "..")          # make the copilot's `app` package importable
os.chdir("..")                    # run from copilot/, like the app does
print("Models are called at:", os.getenv("BIFROST_URL"), "| profile:", os.getenv("SETTINGS_PROFILE") or "direct to Ollama")

## 1. The app asks for a *role*, never a model name
`config/settings.yaml` maps each role to a model. Swapping a model is a config change, not a code change.

In [ ]:
from app.core.config import settings
{role: cfg["model"] for role, cfg in settings()["models"].items()}

## 2. Send one prompt through the gateway

In [ ]:
from app.core import gateway, tracing
reply = gateway.chat("analyst", "In one sentence: what does a payroll provider do for its clients?")
tracing.flush()        # send the trace to Langfuse now
print(reply)

## 3. Find the call
- **Bifrost** → http://localhost:8080 → *Logs*: which model answered and how long it took.
- **Langfuse** → http://localhost:3000 → project *Contract Copilot* → *Tracing*: the `llm.chat` span with the prompt and the reply.

## 4. The gateway refuses models that are not on its allowlist
`infra/bifrost/config.json` lists the only models allowed. Ask for another one:

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=os.getenv("BIFROST_URL"), api_key="local")
try:
    client.chat.completions.create(model="ollama/llama3.2:1b", messages=[{"role": "user", "content": "hi"}])
except Exception as exc:
    print("Refused:", str(exc)[:160])

**Takeaway:** one door to the models, every call visible, and nothing outside the allowlist can run.